In [ ]:
# Cell 1: Authenticate Earth Engine, import dependencies, and prepare the project input folder.
import json
from contextlib import ExitStack
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from shapely.geometry import Point

ee.Authenticate()
ee.Initialize()

cwd = Path.cwd()
project_dir = cwd.parent if cwd.name.lower() == "notebooks" else cwd
input_dir = project_dir / "Input Data"
input_dir.mkdir(parents=True, exist_ok=True)
print(f"Input data folder: {input_dir.resolve()}")

In [ ]:
# Cell 2: Explain how to draw a buffered rectangular AOI and outlet point.
from IPython.display import Markdown, display

display(Markdown("""
## Before you draw

Draw **one rectangle** that fully contains the catchment and leaves a generous buffer on every side, then draw **one outlet point** inside that rectangle. The buffer is important: the final DEM is trimmed to remove projected edge NoData, and a larger extent also prevents the catchment from being clipped during delineation. Draw both shapes on the map below before running the next cell. Large areas are downloaded in smaller tiles automatically.
"""))

In [ ]:
# Cell 3: Load GLO-30 and open the interactive map for drawing the AOI and outlet.
Map = geemap.Map(center=[20, 0], zoom=2)
glo = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
proj = glo.first().projection()
dem = glo.mosaic().setDefaultProjection(proj)
Map.addLayer(
    dem,
    {"min": 0, "max": 8000, "palette": ["#173b8f", "#239bd1", "#67bd63", "#f6e65c", "#e88232", "#f4f1e8"]},
    "GLO-30 elevation",
)
Map

In [ ]:
# Cell 4: Validate and save the user-drawn watershed rectangle and outlet point.
drawn = Map.user_rois
if drawn is None:
    raise ValueError("Draw one rectangle and one outlet point on the map, then rerun this cell.")

features = drawn.getInfo().get("features", [])
polygon_features = [f for f in features if f.get("geometry", {}).get("type") in ("Polygon", "MultiPolygon")]
point_features = [f for f in features if f.get("geometry", {}).get("type") == "Point"]
if len(polygon_features) != 1 or len(point_features) != 1:
    raise ValueError(
        f"Expected exactly one rectangle and one point; found {len(polygon_features)} polygon(s) and {len(point_features)} point(s)."
    )

aoi_geometry = polygon_features[0]["geometry"]
outlet_geometry = point_features[0]["geometry"]
aoi = ee.Geometry(aoi_geometry)

with open(input_dir / "watershedAOI.geojson", "w", encoding="utf-8") as f:
    json.dump(
        {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"name": "watershed"}, "geometry": aoi_geometry}]},
        f,
        indent=2,
    )

outlet_coords = outlet_geometry["coordinates"]
outlet_gdf = gpd.GeoDataFrame(
    {"name": ["outlet"]},
    geometry=[Point(outlet_coords[0], outlet_coords[1])],
    crs="EPSG:4326",
)
outlet_path = input_dir / "outlet.shp"
outlet_gdf.to_file(outlet_path)
print(f"AOI saved: {input_dir / 'watershedAOI.geojson'}")
print(f"Outlet saved: {outlet_path}")

In [ ]:
# Cell 5: Download the AOI DEM, splitting and mosaicking tiles when needed for large areas.
source_dem_path = input_dir / "sourceDEM.tif"
for old_file in [source_dem_path, *input_dir.glob("sourceDEM_*.tif")]:
    if old_file.exists():
        old_file.unlink()


def export_tile(region, path, label, depth=0, max_depth=6):
    try:
        geemap.ee_export_image(
            dem.clip(region),
            filename=str(path),
            scale=30,
            region=region,
            file_per_band=False,
        )
        with rasterio.open(path) as src:
            if src.width == 0 or src.height == 0:
                raise ValueError(f"The {label} download is empty.")
        print(f"Downloaded {label}: {path.name}")
        return [path]
    except Exception as error:
        if path.exists():
            path.unlink()
        if depth >= max_depth:
            raise RuntimeError(f"Could not download {label} after {max_depth} subdivisions.") from error
        print(f"{label} download failed; subdividing it (level {depth + 1}/{max_depth}): {error}")
        bounds = region.bounds(maxError=1).coordinates().getInfo()[0]
        xmin = min(point[0] for point in bounds)
        xmax = max(point[0] for point in bounds)
        ymin = min(point[1] for point in bounds)
        ymax = max(point[1] for point in bounds)
        if xmin == xmax or ymin == ymax:
            raise RuntimeError(f"Cannot subdivide the zero-area region for {label}.") from error
        xmid = (xmin + xmax) / 2
        ymid = (ymin + ymax) / 2
        quadrants = [
            ("NW", ee.Geometry.Rectangle([xmin, ymid, xmid, ymax])),
            ("NE", ee.Geometry.Rectangle([xmid, ymid, xmax, ymax])),
            ("SW", ee.Geometry.Rectangle([xmin, ymin, xmid, ymid])),
            ("SE", ee.Geometry.Rectangle([xmid, ymin, xmax, ymid])),
        ]
        paths = []
        for suffix, subregion in quadrants:
            subpath = path.with_name(f"{path.stem}_{suffix}{path.suffix}")
            paths.extend(export_tile(subregion, subpath, f"{label}-{suffix}", depth + 1, max_depth))
        return paths


try:
    print("Attempting one DEM download for the full rectangle...")
    downloaded_tiles = export_tile(aoi, source_dem_path, "watershed AOI")
except RuntimeError:
    raise
except Exception as error:
    raise RuntimeError("DEM download failed; check Earth Engine access and the selected rectangle.") from error

if downloaded_tiles != [source_dem_path]:
    with ExitStack() as stack:
        datasets = [stack.enter_context(rasterio.open(path)) for path in downloaded_tiles]
        merge(datasets, dst_path=str(source_dem_path))
    print(f"Merged {len(downloaded_tiles)} downloaded tiles into {source_dem_path.name}.")

with rasterio.open(source_dem_path) as src:
    print(f"Source DEM: {source_dem_path}")
    print("CRS        :", src.crs)
    print("Dimensions :", src.width, "x", src.height)
    print("Resolution:", src.res)
    print("Bounds     :", src.bounds)
    print("NoData     :", src.nodata)